About the dataset:
1.id:unique id for a new article
2.title:The title of new article
3.author:Author of the news article
4.text:the text  of the article could be incomplete
5.label:A label that marks wheather the news areticle is real or fake
      

    1:Fake news
    0:real news

In [1]:

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Dataset splitting
from sklearn.model_selection import train_test_split

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Display word clouds
from wordcloud import WordCloud

# Ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")


ModuleNotFoundError: No module named 'nltk'

In [ ]:
#data preprocessing
#load dataset

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake['label'] = 1
true['label'] = 0

df = pd.concat([fake, true], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Fake.csv'

: 

In [ ]:
#Separate input and output features
X = df['cleaned_text']
y = df['label']

print("Input features shape:", X.shape)
print("Target variable shape:", y.shape)

In [ ]:
#Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
#Convert text into numerical features using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

In [ ]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully!")

In [ ]:
#Make predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

print("Predictions:")
print(y_pred_lr[:10])

In [ ]:
#Evaluate the model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", accuracy * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
#Display confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

In [ ]:
#Multinomial Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes Accuracy:",
      accuracy_score(y_test, y_pred_nb) * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

In [ ]:
#andom Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)

y_pred_rf = rf_model.predict(X_test_tfidf)

print("Random Forest Accuracy:",
      accuracy_score(y_test, y_pred_rf) * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
#Support Vector Machine (SVM)
from sklearn.svm import LinearSVC

svm_model = LinearSVC(random_state=42)

svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_test_tfidf)

print("SVM Accuracy:",
      accuracy_score(y_test, y_pred_svm) * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

In [ ]:
#Compare all models
model_results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Naive Bayes',
        'Random Forest',
        'SVM'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_svm)
    ]
})

model_results['Accuracy (%)'] = model_results['Accuracy'] * 100

display(model_results[['Model', 'Accuracy (%)']])

In [ ]:
# Visualize and identify the best-performing model
import matplotlib.pyplot as plt

model_results = model_results.sort_values(
    by='Accuracy (%)',
    ascending=False
).reset_index(drop=True)

plt.figure(figsize=(8, 5))
bars = plt.bar(model_results['Model'], model_results['Accuracy (%)'],
               color='steelblue')

plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.xticks(rotation=20, ha='right')

for bar, accuracy in zip(bars, model_results['Accuracy (%)']):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{accuracy:.2f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

best_model = model_results.iloc[0]
print(f"Best model: {best_model['Model']} "
      f"({best_model['Accuracy (%)']:.2f}% accuracy)")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

predictions_by_model = {
    'Logistic Regression': y_pred_lr,
    'Naive Bayes': y_pred_nb,
    'Random Forest': y_pred_rf,
    'SVM': y_pred_svm
}

best_predictions = predictions_by_model[best_model['Model']]

print(f"Classification report for {best_model['Model']}:")
print(classification_report(y_test, best_predictions, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    cmap='Blues',
    values_format='d'
)
plt.title(f"Confusion Matrix - {best_model['Model']}")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Final metric summary for the best model
best_predictions = predictions_by_model[best_model['Model']]

print("Accuracy:", accuracy_score(y_test, best_predictions))
print("Precision:", precision_score(y_test, best_predictions, average='weighted', zero_division=0))
print("Recall:", recall_score(y_test, best_predictions, average='weighted', zero_division=0))
print("F1-score:", f1_score(y_test, best_predictions, average='weighted', zero_division=0))

# Predict a new sample with the best model
new_article = ["Breaking: local officials confirm a major new policy to reduce inflation and improve wages"]

# If you used a pipeline:
# best_model_pipeline = best_model_pipeline
# new_vector = best_model_pipeline.named_steps['tfidf'].transform(new_article)
# prediction = best_model_pipeline.named_steps['clf'].predict(new_vector)

# If you used a separate vectorizer object:
new_vector = vectorizer.transform(new_article)
prediction = best_classifier.predict(new_vector)

label = "REAL NEWS" if prediction[0] == 0 else "FAKE NEWS"
probability = best_classifier.predict_proba(new_vector)[0]
confidence = max(probability) * 100

print("\nNew Article Prediction:")
print("Label:", label)
print("Confidence:", round(confidence, 2), "%")

In [ ]:
# Final summary table
summary = pd.DataFrame({
    'Model': model_results['Model'],
    'Accuracy (%)': model_results['Accuracy (%)']
})

print("\nFinal Model Summary:")
display(summary.sort_values(by='Accuracy (%)', ascending=False).reset_index(drop=True))

# Optional: save the best model
import joblib

best_model_name = best_model['Model']
best_classifier = {
    'Logistic Regression': lr_model,
    'Naive Bayes': nb_model,
    'Random Forest': rf_model,
    'SVM': svm_model
}[best_model_name]

joblib.dump(best_classifier, 'best_fakenews_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print(f"\nBest model saved as: best_fakenews_model.pkl")
print(f"Vectorizer saved as: tfidf_vectorizer.pkl")

In [ ]:
# Final summary table
summary = pd.DataFrame({
    'Model': model_results['Model'],
    'Accuracy (%)': model_results['Accuracy (%)'],
    'Precision (%)': model_results['Precision (%)'],
    'Recall (%)': model_results['Recall (%)'],
    'F1 Score (%)': model_results['F1 Score (%)']
})

# Sort by accuracy (highest first)
summary = summary.sort_values(by='Accuracy (%)', ascending=False).reset_index(drop=True)

# Add rank column
summary.insert(0, 'Rank', range(1, len(summary) + 1))

print("\nFinal Model Summary:")
display(summary)

# Best model selection
best_model_name = summary.iloc[0]['Model']
print(f"\nBest Model: {best_model_name}")

# Map model name to model object
model_dict = {
    'Logistic Regression': lr_model,
    'Naive Bayes': nb_model,
    'Random Forest': rf_model,
    'SVM': svm_model
}

best_classifier = model_dict[best_model_name]

# Save the best model and vectorizer
import joblib

joblib.dump(best_classifier, 'best_fakenews_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("\nBest model and vectorizer saved successfully.")
print("Files created: best_fakenews_model.pkl and tfidf_vectorizer.pkl")

In [ ]:
# Final summary table
summary = pd.DataFrame({
    'Model': model_results['Model'],
    'Accuracy (%)': model_results['Accuracy (%)']
})

summary = summary.sort_values(by='Accuracy (%)', ascending=False).reset_index(drop=True)
summary.insert(0, 'Rank', range(1, len(summary) + 1))

print("\nFinal Model Summary:")
display(summary)

best_model_name = summary.iloc[0]['Model']
print(f"\nBest model: {best_model_name}")

In [ ]:
import matplotlib.pyplot as plt

print("\nFinal Model Summary:")
print(summary.to_string(index=False))

best_model = summary.iloc[0]
print(
    f"\nBest model: {best_model['Model']} "
    f"({best_model['Accuracy (%)']:.2f}% accuracy)"
)

# Compare model accuracy
plt.figure(figsize=(10, 6))
bars = plt.barh(
    summary["Model"],
    summary["Accuracy (%)"],
    color="steelblue"
)

plt.xlabel("Accuracy (%)")
plt.ylabel("Model")
plt.title("Fake News Classification Model Comparison")
plt.xlim(0, 100)
plt.gca().invert_yaxis()

for bar in bars:
    value = bar.get_width()
    plt.text(value + 0.5, bar.get_y() + bar.get_height() / 2,
             f"{value:.2f}%", va="center")

plt.tight_layout()
plt.show()

# Save the ranked results
summary.to_csv("model_accuracy_summary.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

print("\nFinal Model Summary:")
print(summary.to_string(index=False))

best_model = summary.iloc[0]
print(
    f"\nBest model: {best_model['Model']} "
    f"({best_model['Accuracy (%)']:.2f}% accuracy)"
)

plt.figure(figsize=(10, 6))
bars = plt.barh(summary["Model"], summary["Accuracy (%)"], color="steelblue")

plt.xlabel("Accuracy (%)")
plt.ylabel("Model")
plt.title("Fake News Classification Model Comparison")
plt.xlim(0, 100)
plt.gca().invert_yaxis()

for bar in bars:
    value = bar.get_width()
    plt.text(
        value + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

summary.to_csv("model_accuracy_summary.csv", index=False)